<a href="https://colab.research.google.com/github/darrickpang/Email/blob/master/Deep_learning_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Set up and GPU

In [ ]:
!nvidia-smi
import os, sys, time, json, math, glob
from pathlib import Path

ROOT = Path("/content/av_perception")
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
print("Working dir:", ROOT)

### Install dependencies

In [ ]:
!pip -q install ultralytics==8.* fiftyone opencv-python imageio imageio-ffmpeg tqdm

import ultralytics
print("Ultralytics:", ultralytics.__version__)

### Download BDD100K

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz

# You can increase these once it works (e.g., 5000/1000)
MAX_TRAIN = 2000
MAX_VAL   = 400

CLASSES = [
    "car", "bus", "truck", "person", "bike", "motor"
]

# Load a subset of BDD100K with detection labels
train_ds = foz.load_zoo_dataset(
    "bdd100k",
    split="train",
    label_types=["detections"],
    max_samples=MAX_TRAIN,
)

val_ds = foz.load_zoo_dataset(
    "bdd100k",
    split="validation",
    label_types=["detections"],
    max_samples=MAX_VAL,
)

print("Train samples:", len(train_ds))
print("Val samples:", len(val_ds))

### Keep classes

In [ ]:
from fiftyone import ViewField as F

# BDD100K label field name in FiftyOne zoo is usually "detections"
LABEL_FIELD = "detections"

def filter_classes(ds, classes):
    view = ds.filter_labels(LABEL_FIELD, F("label").is_in(classes))
    return view

train_view = filter_classes(train_ds, CLASSES)
val_view   = filter_classes(val_ds, CLASSES)

print("Filtered train:", len(train_view))
print("Filtered val:", len(val_view))

### Export to YOLO format

In [ ]:
import yaml
from pathlib import Path

EXPORT_DIR = ROOT / "bdd100k_yolo"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Export YOLOv5 format (Ultralytics YOLOv8 reads it fine)
train_view.export(
    export_dir=str(EXPORT_DIR),
    dataset_type=fo.types.YOLOv5Dataset,
    label_field=LABEL_FIELD,
    split="train",
    classes=CLASSES,
)

val_view.export(
    export_dir=str(EXPORT_DIR),
    dataset_type=fo.types.YOLOv5Dataset,
    label_field=LABEL_FIELD,
    split="val",
    classes=CLASSES,
)

data_yaml = {
    "path": str(EXPORT_DIR),
    "train": "train/images",
    "val": "val/images",
    "names": {i: c for i, c in enumerate(CLASSES)},
}

DATA_YAML_PATH = ROOT / "bdd100k_subset.yaml"
with open(DATA_YAML_PATH, "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print("Wrote:", DATA_YAML_PATH)
print(data_yaml)

### Train YOLO

In [ ]:
from ultralytics import YOLO

MODEL_NAME = "yolov8n.pt"  # try yolov8s.pt later if you want stronger results
model = YOLO(MODEL_NAME)

RUN_NAME = "bdd100k_subset_yolov8n"

results = model.train(
    data=str(DATA_YAML_PATH),
    epochs=20,
    imgsz=640,
    batch=16,          # if you hit OOM, reduce to 8
    device=0,
    workers=2,
    project=str(ROOT / "runs"),
    name=RUN_NAME,
    patience=7,
    verbose=True,
)

### Validate model

In [ ]:
best_pt = ROOT / "runs" / "detect" / RUN_NAME / "weights" / "best.pt"
print("Best weights:", best_pt)

model = YOLO(str(best_pt))
val_metrics = model.val(data=str(DATA_YAML_PATH), imgsz=640, device=0)
print(val_metrics)

### Visualize predictions

In [ ]:
import random
import cv2

pred_dir = ROOT / "pred_samples"
pred_dir.mkdir(exist_ok=True)

# Grab a few val images
val_images = list((EXPORT_DIR / "val" / "images").glob("*.jpg"))
sample_imgs = random.sample(val_images, k=min(12, len(val_images)))

for p in sample_imgs:
    out = model.predict(source=str(p), imgsz=640, conf=0.25, save=True, project=str(pred_dir), name="pred", verbose=False)

print("Saved predictions to:", pred_dir / "pred")

### Download sample dashcam video

In [ ]:
# Ultralytics sample traffic video
!wget -q -O traffic.mp4 https://github.com/ultralytics/assets/releases/download/v0.0.0/traffic.mp4
print("Downloaded traffic.mp4")

### Run detection

In [ ]:
import cv2
import numpy as np
from tqdm import tqdm

VIDEO_IN  = str(ROOT / "traffic.mp4")
VIDEO_OUT = str(ROOT / "traffic_ttc_out.mp4")

# --- Simple camera-distance heuristic ---
# Assume average object real height (meters). These are rough.
REAL_HEIGHT = {
    "person": 1.7,
    "car": 1.5,
    "bus": 3.2,
    "truck": 3.0,
    "bike": 1.2,
    "motor": 1.4,
}

# Assume focal length in "pixel units" for a 640-ish frame.
# This is not calibrated, but works to illustrate the pipeline.
FOCAL_PX = 700.0

TTC_ALERT_SEC = 2.0
CONF_THRES = 0.25

cap = cv2.VideoCapture(VIDEO_IN)
fps = cap.get(cv2.CAP_PROP_FPS)
W   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(VIDEO_OUT, fourcc, fps, (W, H))

# Track history: track_id -> (prev_distance_m, prev_time)
track_state = {}

frame_idx = 0
pbar = tqdm(total=n_frames, desc="Tracking+TTC")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    t_sec = frame_idx / fps

    # Track with YOLOv8 (ByteTrack)
    # persist=True keeps IDs stable across frames
    tr = model.track(
        source=frame,
        persist=True,
        imgsz=640,
        conf=CONF_THRES,
        verbose=False
    )[0]

    # Draw and compute TTC
    overlay = frame.copy()

    if tr.boxes is not None and len(tr.boxes) > 0:
        boxes = tr.boxes.xyxy.cpu().numpy()
        clss  = tr.boxes.cls.cpu().numpy().astype(int)
        confs = tr.boxes.conf.cpu().numpy()
        ids   = None
        if tr.boxes.id is not None:
            ids = tr.boxes.id.cpu().numpy().astype(int)

        for i in range(len(boxes)):
            x1, y1, x2, y2 = boxes[i].astype(int)
            c = clss[i]
            conf = float(confs[i])
            label = CLASSES[c] if c < len(CLASSES) else str(c)
            track_id = int(ids[i]) if ids is not None else -1

            # Distance estimate
            bbox_h_px = max(1, (y2 - y1))
            real_h = REAL_HEIGHT.get(label, 1.6)
            dist_m = (real_h * FOCAL_PX) / bbox_h_px

            # Relative speed estimate from distance derivative
            ttc = None
            if track_id != -1:
                if track_id in track_state:
                    prev_dist, prev_t = track_state[track_id]
                    dt = max(1e-3, (t_sec - prev_t))
                    closing_speed = (prev_dist - dist_m) / dt  # >0 means getting closer
                    if closing_speed > 0.1:  # avoid noise
                        ttc = dist_m / closing_speed
                track_state[track_id] = (dist_m, t_sec)

            # Draw box
            cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 255), 2)

            # Text
            txt1 = f"{label} id={track_id} {conf:.2f}"
            txt2 = f"dist~{dist_m:.1f}m"
            if ttc is not None and ttc < 99:
                txt2 += f" TTC~{ttc:.1f}s"

            cv2.putText(overlay, txt1, (x1, max(20, y1 - 10)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
            cv2.putText(overlay, txt2, (x1, min(H - 10, y2 + 20)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

            # Near-miss alert
            if ttc is not None and ttc < TTC_ALERT_SEC:
                cv2.putText(overlay, "NEAR-MISS RISK!",
                            (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1.1, (0, 0, 255), 3)
                cv2.putText(overlay, f"TTC~{ttc:.2f}s",
                            (20, 80), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 3)

    writer.write(overlay)
    frame_idx += 1
    pbar.update(1)

pbar.close()
cap.release()
writer.release()

print("Saved:", VIDEO_OUT)

### Convert to GIF

In [ ]:
import imageio.v2 as imageio

gif_out = str(ROOT / "traffic_ttc_out.gif")

reader = imageio.get_reader(VIDEO_OUT)
fps_gif = 10  # lower for smaller file
frames = []
max_frames = 180  # ~18 seconds at 10 fps (adjust)

for i, frame in enumerate(reader):
    if i % int(max(1, (reader.get_meta_data()['fps'] // fps_gif))) == 0:
        frames.append(frame)
    if len(frames) >= max_frames:
        break

imageio.mimsave(gif_out, frames, fps=fps_gif)
print("Saved:", gif_out)

### Display results on Colab

In [ ]:
from IPython.display import Video, Image, display

display(Video(VIDEO_OUT, embed=True))
display(Image(filename=gif_out))